In [50]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year


Successfully saved authorization token.


Show how the standard deviation increases with biomass and cannot be used to filter data by quality

In [52]:
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").rename("age")

biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').mean().select("AGB").rename("biomass")
sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))

## Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again


# Export the data with GEDI asymptote, GEDI biomass and with the age pixels only with contiguous patches of >1ha of area

In [54]:
fire = (ee.Image("projects/mapbiomas-public/assets/brazil/fire/collection3/mapbiomas_fire_collection3_annual_burned_coverage_v1")
    .select([f"burned_coverage_{year}" for year in config.range_1985_2020])
    .byte()
    .rename([str(year) for year in config.range_1985_2020])
    .gt(0)
    .reduce('sum').rename("num_fires")).unmask(0)

floodable_forests = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
        .select(f"classification_{last_year}").eq(6)).rename("floodable_forests")

### ----------------- Surrounding Landscape -----------------

quarters_ecoreg_biomass = ee.Image("projects/forestregrowth/assets/quarters_ecoreg_biomass")
ecoreg = ee.Image("projects/forestregrowth/assets/ecoreg")
distance_deep_forest = ee.Image(f"{data_folder}/distance_deep_forest").rename("dist")
sur_cover = ee.Image(f"{data_folder}/sur_cover")
nearest_mature = ee.Image("projects/forestregrowth/assets/nearest_mature_GEDI")

### ----------------- Environmental -----------------

categorical = ee.Image(f"{data_folder}/categorical")
topography = ee.Image("CSP/ERGo/1_0/Global/ALOS_landforms").rename("topography") # 90m resolution
soil = ee.Image(f"{data_folder}/soilgrids")
terraclim = ee.Image(f"{data_folder}/terraclim_1958_2019")

In [55]:
def reduce_reproject(image, reducer):
    """Reproject image using mean."""
    return image.reduceResolution(
        reducer=reducer
    ).reproject(
        crs=age.projection(),
        scale=age.projection().nominalScale()
    )


# Reproject continuous variables using mean
continuous_vars = [
    nearest_mature.rename("nearest_mature"), # double
    soil, # float
    terraclim.select(["mean_soil", "mean_vpd", "mean_temp", "mean_def", 
                      "mean_srad", "mean_pr", "mean_pdsi", "mean_aet"]) # float and int16
]

# Reproject categorical variables using first
categorical_vars = [
    fire, # int64
    ecoreg, # int16
    categorical, # int8
    topography # int8
]

# Create unified dataset by directly combining bands
unified_data = ee.Image.cat([
    age, # int8
    floodable_forests,
    distance_deep_forest, # int16
    sur_cover, # float
    *[reduce_reproject(var, ee.Reducer.mean()) for var in continuous_vars],
    *[reduce_reproject(var, ee.Reducer.first()) for var in categorical_vars],
    ee.Image.pixelLonLat().rename(['lon', 'lat'])
])

In [79]:
def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2020-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI = GEDI.mosaic().toInt16().rename('biomass')
GEDI_reproj = GEDI.reproject(age.projection())


# unified_data_secondary = ee.Image.cat([
#     unified_data,
#     GEDI
# ])

GEDI_mask = GEDI_reproj.gt(0).selfMask().rename("mask")

unified_data = ee.Image.cat([age, GEDI_reproj, biomass.rename("ESA_biomass"), GEDI_mask])

biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")

unified_data = unified_data.updateMask(biomes.eq(1))#.updateMask(GEDI_mask)

# map = geemap.Map()
# map.addLayer(unified_data, {}, 'u')
# map

In [80]:

# Sample within the tile geometry
unified_data_sampled = unified_data.stratifiedSample(
    numPoints = 10000, classBand='mask', geometries=False
)

# Filter properties to export
to_remove = ['.geo', 'system:index']
all_properties = unified_data.bandNames().getInfo()
properties_to_export = [p for p in all_properties if p not in to_remove]

# Export task to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=unified_data_sampled,
    description='age_gedi_esa',
    folder = 'gedi_esa_comparison',
    fileFormat='CSV',
    selectors=properties_to_export
)
task.start()

In [60]:
grid = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed_gedi")

# grid = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_edge_removed_gedi")

def export_csv(name, image, grid_size, n_chunks = 1, lu_name = None):

    properties_to_export = image.bandNames().getInfo()
    
    total_features = grid.size().getInfo()
    chunk_size = int(total_features * 1/n_chunks)

    def process_chunk(chunk_index):
        start = chunk_index * chunk_size
        chunk = grid.toList(chunk_size, start)
        selected_pixels = ee.FeatureCollection(chunk)

        unified_fc = image.reduceRegions(selected_pixels, ee.Reducer.first(), 30)

        task = ee.batch.Export.table.toDrive(
            collection = unified_fc,
            description = f"grid_{grid_size}k_{name}_{chunk_index}_gedi",
            fileFormat = "CSV",
            selectors = [p for p in properties_to_export if p not in ['system:index', '.geo']]
        )
        task.start()

    for i in range(n_chunks):
        if i*chunk_size < total_features:
            process_chunk(i)

export_csv("secondary", unified_data_secondary, 10, n_chunks = 30)

## Export age and biomass for ESA CCI for the 1ha patches

In [92]:

# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    ).start()

